In [11]:
import json
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
from typing import List, Dict, Tuple, Any
import re
import os
from glob import glob
from collections import defaultdict, Counter
from statistics import mean
import ast
import math
import numpy as np
import csv
from urllib.parse import urlparse

# RQ1: Can LLM-Driven Agents Reliably Audit Dark Patterns?

## Table 2: Overall Performance Across Prompting Strategies. Accuracy for each prompting strategy is shown below.


In [12]:
df = pd.read_csv('../results/prompt_ablation/with_raw_verfication.csv')
df = df[df['task_completion'] != "Incomplete"]
cols_to_keep = [
    "data_broker",
    "model",
    "prompting_strategy",
    "combined_prompting_strategy",
    "pattern_name",
    "status",
    "classification_accuracy",
    "explanation_accuracy",
    "is_correct_classification",
    "model_found",
    "actual_found"
]

df = df[cols_to_keep]

# rename column 'combined_prompting_strategy': 'baseline_zero_shot' -> 'Baseline Zero-Shot', 'zero_shot_regulator_role' -> 'Zero-Shot Regulator Role', 'regulator_role_few_shot_scenarios' -> 'Regulator Role Few-Shot Scenarios', 'few_shot_regulator_role_with_CoT' -> 'Few-Shot Regulator Role with CoT'
df["combined_prompting_strategy"] = df["combined_prompting_strategy"].replace({
    "baseline_zero_shot": "Zero shot",
    "zero_shot_regulator_role": "Zero shot + Role",
    "regulator_role_few_shot_scenarios": "Few shot + Role",
    "few_shot_regulator_role_with_CoT": "Few shot + Role + CoT"
})

# only consider eight pattern_name
pattern_names = ['Creating Barriers', 'Adding Steps', 'Privacy Mazes', 'Visual Prominence',
                 'Hidden Information', 'Feedforward Ambiguity', 'Conflicting Information', 'Information Without Context']

df = df[df['pattern_name'].isin(pattern_names)]

strategy_order = [
    "Zero shot",
    "Zero shot + Role",
    "Few shot + Role",
    "Few shot + Role + CoT"
]
cat_type = pd.CategoricalDtype(strategy_order, ordered=True)
df["combined_prompting_strategy"] = df["combined_prompting_strategy"].astype(cat_type)

In [13]:
def compute_metrics(group):   
    total = len(group)
    predicted = group["model_found"]
    ground_truth = group["actual_found"]

    # Classification metrics
    # -------------------------
    TP = (predicted & ground_truth).sum()
    FP = (predicted & ~ground_truth).sum()
    FN = (~predicted & ground_truth).sum()
    TN = (~predicted & ~ground_truth).sum()


    cls_accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    # -------------------------
    # Explanation (conditional only for TP cases)
    is_TP = predicted & ground_truth
    is_exp_correct = group["explanation_accuracy"].isin(["Correct", "Partial"])
    TP_count = is_TP.sum()
    explanation_accuracy = (
        (is_TP & is_exp_correct).sum() / TP_count
        if TP_count else 0
    )

    return pd.Series({
        "total_samples": total,
        "actual_found": ground_truth.sum(),
        "actual_found_percentage": ground_truth.sum() / total * 100 if total else 0,
        "predicted_found": predicted.sum(),
        "predicted_found_percentage": predicted.sum() / total * 100 if total else 0,
        
        #Classification
        "cls_TP": TP,
        "cls_FP": FP,
        "cls_FN": FN,
        "cls_TN": TN,
        "cls_accuracy": cls_accuracy * 100,
        "cls_precision": precision * 100,
        "cls_recall": recall * 100,
        "cls_f1": f1 * 100,

        # Explanation
        "explanation_accuracy": explanation_accuracy * 100,
    })


### table2 upper section

In [14]:
metrics_summary_per_strategy = (
    df.groupby("combined_prompting_strategy")
    .apply(compute_metrics)
    .sort_index()
    .round(2)
)

columns_to_display = [
    'cls_accuracy',  'cls_precision',  'cls_recall', 'cls_f1', 'explanation_accuracy'
]
# only display 2 decimal places for the metrics
latex = metrics_summary_per_strategy[columns_to_display].to_latex(
    index=True,
    float_format="%.1f",
    caption="Metrics summary per strategy",
    label="tab:prompting_accuracy",
    position="t"
)

print(latex)

\begin{table}[t]
\caption{Metrics summary per strategy}
\label{tab:prompting_accuracy}
\begin{tabular}{lrrrrr}
\toprule
 & cls_accuracy & cls_precision & cls_recall & cls_f1 & explanation_accuracy \\
combined_prompting_strategy &  &  &  &  &  \\
\midrule
Zero shot & 70.8 & 61.0 & 60.1 & 60.5 & 78.1 \\
Zero shot + Role & 63.6 & 49.9 & 73.3 & 59.4 & 71.1 \\
Few shot + Role & 83.5 & 79.9 & 74.0 & 76.9 & 95.8 \\
Few shot + Role + CoT & 86.7 & 88.0 & 74.4 & 80.7 & 98.5 \\
\bottomrule
\end{tabular}
\end{table}



/var/folders/19/d351fdkn37136xmwl_hyygww0000gn/T/ipykernel_8573/842167358.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("combined_prompting_strategy")
/var/folders/19/d351fdkn37136xmwl_hyygww0000gn/T/ipykernel_8573/842167358.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_metrics)


### table2 lower section

In [14]:

# One-sided p-value tests whether B strictly improves over A. 95% CI interpretation 
 
def compute_metrics_bootstrap(group):
    total = len(group)
    predicted = group["model_found"]
    ground_truth = group["actual_found"]

    TP = (predicted & ground_truth).sum()
    FP = (predicted & ~ground_truth).sum()
    FN = (~predicted & ground_truth).sum()
    TN = (~predicted & ~ground_truth).sum()

    cls_accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    is_TP = predicted & ground_truth
    is_exp_correct = group["explanation_accuracy"].isin(["Correct", "Partial"])
    TP_count = is_TP.sum()
    explanation_accuracy = (is_TP & is_exp_correct).sum() / TP_count if TP_count else 0

    return {
        "cls_accuracy": cls_accuracy * 100,
        "cls_precision": precision * 100,
        "cls_recall": recall * 100,
        "cls_f1": f1 * 100,
        "explanation_accuracy": explanation_accuracy * 100,
    }



def bootstrap_strategy(df, strategy, n_brokers=50, n_iterations=100, seed=42):
    """
    For each strategy, bootstrap_strategy does this 1000 times
    Randomly samples 50 brokers with replacement
    Takes all rows belonging to those 50 brokers
    compute metrics for classification and explanation accuracy
    """
    rng = np.random.default_rng(seed)
    subset = df[df["combined_prompting_strategy"] == strategy]
    brokers = subset["data_broker"].unique()

    records = []
    for _ in range(n_iterations):
        sampled_brokers = rng.choice(brokers, size=n_brokers, replace=True)
        # Build resampled dataframe (handle duplicate broker names)
        sampled_dfs = [subset[subset["data_broker"] == b] for b in sampled_brokers]
        boot_df = pd.concat(sampled_dfs, ignore_index=True)
        records.append(compute_metrics_bootstrap(boot_df))

    return pd.DataFrame(records)


def bootstrap_comparison(df, strategy_a, strategy_b, metric="cls_recall",
                          n_brokers=50, n_iterations=100, seed=42):
    """
    Compare two strategies on a metric using bootstrap.
    Returns bootstrap distributions, observed difference, and p-value.
    
    H0: metric(strategy_b) - metric(strategy_a) <= 0  (no improvement)
    p-value = proportion of bootstrap samples where diff <= 0
    """
    dist_a = bootstrap_strategy(df, strategy_a, n_brokers, n_iterations, seed)
    dist_b = bootstrap_strategy(df, strategy_b, n_brokers, n_iterations, seed + 1)
    
    # This subtracts the two distributions element-wise, giving 1000 difference values.
    # in this bootstrap sample, how much better was strategy B than A
    diff = dist_b[metric] - dist_a[metric]

    # This subtracts the two distributions element-wise, giving 1000 difference values. 
    # For each of the 1000 bootstrap samples, in this bootstrap sample, how much better was strategy B than A
    # Observed (non-bootstrapped) metrics for reference
    obs_a = compute_metrics_bootstrap(df[df["combined_prompting_strategy"] == strategy_a])
    obs_b = compute_metrics_bootstrap(df[df["combined_prompting_strategy"] == strategy_b])
    observed_diff = obs_b[metric] - obs_a[metric]

    # One-sided p-value: P(diff <= 0) under bootstrap
    p_value = (diff <= 0).mean()
    
    # Take 2.5th and 97.5th percentile of those 1000 difference values
    ci_lower, ci_upper = np.percentile(diff, [2.5, 97.5])

    return {
        "strategy_a": strategy_a,
        "strategy_b": strategy_b,
        "metric": metric,
        "observed_a": round(obs_a[metric], 4),
        "observed_b": round(obs_b[metric], 4),
        "observed_diff": round(observed_diff, 4),
        "boot_mean_diff": round(diff.mean(), 4),
        "ci_95_lower": round(ci_lower, 4),
        "ci_95_upper": round(ci_upper, 4),
        "p_value": round(p_value, 4),
        "significant (p<0.05)": p_value < 0.05,
        "dist_a": dist_a[metric].values,  # full distribution if needed
        "dist_b": dist_b[metric].values,
    }


In [ ]:
# Sequential ablation comparisons only 
sequential_pairs = [
    ("Zero shot", "Zero shot + Role"),  # Zero shot → Zero shot + Role
    ("Zero shot + Role", "Few shot + Role"),  # Zero shot + Role → Few shot + Role
    ("Few shot + Role", "Few shot + Role + CoT"),  # Few shot + Role → Few shot + Role + CoT
]

metrics_to_test = ["cls_recall", "cls_precision", "cls_f1", "cls_accuracy", "explanation_accuracy"]

results = []
for metric in metrics_to_test:
    for a, b in sequential_pairs:
        res = bootstrap_comparison(df, a, b, metric=metric, n_brokers=50, n_iterations=1000)
        results.append({k: v for k, v in res.items() if k not in ("dist_a", "dist_b")})

results_df = pd.DataFrame(results)

# Print grouped by metric 
print("\n=== Ablation Study: Sequential Bootstrap Significance Tests ===\n")
for metric in metrics_to_test:
    print(f"── {metric} ──")
    sub = results_df[results_df["metric"] == metric][
        ["strategy_a", "strategy_b", "ci_95_lower", "ci_95_upper", "p_value", "significant (p<0.05)"]
    ]
    print(sub.round(4).to_string(index=False))
    print()


## Table 3: Performance by Dark Pattern Category (only consider prompting strategy 'Few shot + Role + CoT')

In [ ]:
metrics_summary_per_pattern_per_prompting = (
    df.groupby(["pattern_name", "combined_prompting_strategy"])
    .apply(compute_metrics)
    .sort_index()
    .round(1)
)

# only consider prompting strategy 'Few shot + Role + CoT'
df_few_shot_cot = df[df['combined_prompting_strategy'] == 'Few shot + Role + CoT']
metrics_summary_per_pattern = (
    df_few_shot_cot.groupby("pattern_name")
    .apply(compute_metrics)
    .sort_index()
    .round(1)
)
columns_to_display = [
    'cls_accuracy',  'cls_precision',  'cls_recall', 'cls_f1', 'explanation_accuracy'
]
# only display 2 decimal places for the metrics
latex = metrics_summary_per_pattern[columns_to_display].to_latex(
    index=True,
    float_format="%.1f",
    caption="Metrics summary per pattern",
    label="tab:pattern_level_accuracy",
    position="t"
)

print(latex)


## Table 4: Dark Pattern Prevalence Estimates

### Dark pattern prevalence on manual annotated data brokers

In [ ]:
def proportion_ci(p, n):
    """95% binomial CI"""
    if n == 0:
        return (0, 0)
    se = math.sqrt(p * (1 - p) / n)
    margin = 1.96 * se
    return max(0, p - margin), min(1, p + margin)


df_annotated = pd.read_csv('../results/prompt_ablation/with_raw_verfication.csv')
df_annotated_block_type = pd.read_csv('../results/prompt_ablation/with_verification_status.csv')

pattern_names = ['Creating Barriers', 'Adding Steps', 'Privacy Mazes', 'Visual Prominence',
                 'Hidden Information', 'Feedforward Ambiguity', 'Conflicting Information', 'Information Without Context']
df_annotated = df_annotated[df_annotated['pattern_name'].isin(pattern_names)]
df_annotated = df_annotated[df_annotated['combined_prompting_strategy'] == 'Few shot + Role + CoT']

df_annotated = df_annotated[df_annotated['task_completion'] != "Incomplete"]
df_filter = df_annotated.merge(df_annotated_block_type[['data_broker', 'prompting_strategy', 'group_incomplete', 'group_blocked']], on=['data_broker', 'prompting_strategy'], how='left')
#print(df_filter.head())

# remove rows where group_incomplete is True or group_blocked is True
df_filter = df_filter[~((df_filter['group_incomplete'] == True) | (df_filter['group_blocked'] == True))]

# Report the prevalence of each pattern using Model Prediction Results
# for each pattern, how many brokers are found to have this pattern, and how many brokers in total, and the percentage with 95% confidence interval
df_filter["found"] = df_filter["status"].str.lower() == "found"

broker_pattern = (
    df_filter.groupby(["pattern_name", "data_broker"])["found"]
    .any()
    .reset_index()
)
summary = (
    broker_pattern
    .groupby("pattern_name")["found"]
    .agg(
        num_brokers_found="sum",
        total_brokers="count"
    )
)

summary["percentage"] = summary["num_brokers_found"] / summary["total_brokers"] * 100
summary["percentage"] = (
    summary["num_brokers_found"] / summary["total_brokers"] * 100
)
summary["ci_lower"], summary["ci_upper"] = zip(*summary.apply(lambda row: proportion_ci(row["num_brokers_found"] / row["total_brokers"], row["total_brokers"]), axis=1))
summary["ci_lower"] = summary["ci_lower"] * 100
summary["ci_upper"] = summary["ci_upper"] * 100
summary["95%CI"] = summary.apply(
    lambda r: f'[{r["ci_lower"]:.1f}, {r["ci_upper"]:.1f}]',
    axis=1
)

# rename 'num_brokers_found/total_brokers' to 'Detected Brokers (n/N)'
summary = summary.rename(columns={'percentage': 'Prevalence (%)'})
summary = summary.rename(columns={'95%CI': '95%CI (%)'})

#print(summary)

columns_to_display = [
      'Prevalence (%)', '95%CI (%)'
]

# only display 2 decimal places for the metrics
latex = summary[columns_to_display].to_latex(
    index=True,
    float_format="%.1f",
    caption="Dark Pattern Prevalence Based on Model Prediction Results",
    label="tab:prevalence_model_prediction",
    position="t"
)

print(latex)

### Dark pattern prevalence on test data brokers

In [ ]:
# if only consider no_blocking rows, complete remove rows if have block, what is the prevalence?
df_left_block = pd.read_csv('../results/deployment/with_verification_status.csv')
df = df_left_block
pattern_names = ['Creating Barriers', 'Adding Steps', 'Privacy Mazes', 'Visual Prominence',
                 'Hidden Information', 'Feedforward Ambiguity', 'Conflicting Information', 'Information Without Context']
df = df[df['pattern_name'].isin(pattern_names)]

# only keep rows with blocked == False and is_successful == True
df = df[(df['blocked'] == False) & df['is_successful'] == True]

df["found"] = df["status"].str.lower() == "found"
broker_pattern = (
    df.groupby(["pattern_name", "data_broker"])["found"]
    .any()
    .reset_index()
)
summary = (
    broker_pattern
    .groupby("pattern_name")["found"]
    .agg(
        num_brokers_found="sum",
        total_brokers="count"
    )
)

summary["percentage"] = (
    summary["num_brokers_found"] / summary["total_brokers"] * 100
)

summary["percentage"] = summary["num_brokers_found"] / summary["total_brokers"] * 100
summary["percentage"] = (
    summary["num_brokers_found"] / summary["total_brokers"] * 100
)
summary["ci_lower"], summary["ci_upper"] = zip(*summary.apply(lambda row: proportion_ci(row["num_brokers_found"] / row["total_brokers"], row["total_brokers"]), axis=1))
summary["ci_lower"] = summary["ci_lower"] * 100
summary["ci_upper"] = summary["ci_upper"] * 100
summary["95%CI"] = summary.apply(
    lambda r: f'[{r["ci_lower"]:.1f}, {r["ci_upper"]:.1f}]',
    axis=1
)

# rename 'num_brokers_found/total_brokers' to 'Detected Brokers (n/N)'
summary = summary.rename(columns={'percentage': 'Prevalence (%)'})
summary = summary.rename(columns={'95%CI': '95%CI (%)'})

#print(summary)
columns_to_display = [
      'Prevalence (%)', '95%CI (%)'
]

# only display 2 decimal places for the metrics
latex = summary[columns_to_display].to_latex(
    index=True,
    float_format="%.1f",
    caption="Dark Pattern Prevalence Based on Model Prediction Results",
    label="tab:prevalence_model_prediction",
    position="t"
)

print(latex)